In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import LPNE
import AREmulator
import run_config as rc
import statschecker as sc
import standardplots as sp

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42

### Setup

In [ ]:
# config
DATASET = rc.DATASET        # '1461ka'
TRAIN_START = 'mid'         # key of rc.STARTS: 'early', 'mid', 'late', 'balanced' (1461-579 ka, no test set) or 'default' (80:20 split)
N_FEATURES = rc.NFEATURES

PARAMS = {'conv filters': 30, 'conv kernel': 10, 'dense unit': 28, 
          'dropout rate': 0.3353763108415839, 'learning rate': 0.00038902232764492545, 
          'dwt gamma': 0.6506584806454648, 'dwt beta': 0.4637125670415251, 'dwt eps': 4.1664542731895874e-06, 
          'dwt levels': 2} # taken from optimize_model.ipynb, the 'mid' start optimization set

# one config per insolation record: the choice is part of the config (and so of the run tag)
cfgs = {insol: rc.make_config(DATASET, seed=seed, insol=insol, train_start=TRAIN_START, window_kyr = 100)
        for insol in rc.INSOL_COLS}
# and the Ber90 orbital solution decomposed into its elements - eccentricity, precession index (e sin varpi) and
# obliquity - as three forcing columns instead of the one solstice-insolation column (same window, split and params)
cfg_dec = rc.make_config(DATASET, seed=seed, insol='insol_ber90', train_start=TRAIN_START, window_kyr = 100,
                         forcing=('ecc', 'prec', 'obl'))

cfg = rc.make_config(
    DATASET,
    seed=seed,
    params=rc.DEFAULT_PARAMS,   # hyperparameter dict; e.g. an Optuna study's best_params to swap
    train_start=TRAIN_START,
)
paramsa = cfgs['insol_huy06']['params']
pathsa = rc.run_paths(cfgs['insol_huy06']['run_tag'])

print("run tag :", cfgs['insol_huy06']['run_tag'])
print("weights :", pathsa['weights'])
print("params  :", paramsa)

In [ ]:
# load in and split data
raw_huy06 = rc.load_data(cfgs['insol_huy06'])
raw_ber90 = rc.load_data(cfgs['insol_ber90'])
splits_huy06 = rc.split_data(raw_huy06, cfgs['insol_huy06'])
splits_ber90 = rc.split_data(raw_ber90, cfgs['insol_ber90'])
raw_dec = rc.load_data(cfg_dec)    # X, Y, Z, ecc, prec, obl
splits_dec = rc.split_data(raw_dec, cfg_dec)

print(f'using huy06 record: {rc.describe_split(splits_huy06)}')
print(f'using ber90 record: {rc.describe_split(splits_ber90)}')
print(f'using decomposed ber90 record: {rc.describe_split(splits_dec)}')

# time splits same for all three insolation records, since they share the same age column
testt, traint, valt = splits_huy06['test_time'], splits_huy06['train_time'], splits_huy06['validation_time']


In [ ]:
# normalize data
scalea, traina, vala, testa = rc.fit_and_transform_splits(splits_huy06)
scaleb, trainb, valb, testb = rc.fit_and_transform_splits(splits_ber90)
scalec, trainc, valc, testc = rc.fit_and_transform_splits(splits_dec)
# can also get and transform standard deviations of records

# chunk normalized data into windows
X_traina, Y_traina, Insol_traina = LPNE.create_windows(traina, cfgs['insol_huy06']['window_size'], cfgs['insol_huy06']['rollout_length'])
X_vala, Y_vala, Insol_vala = LPNE.create_windows(vala, cfgs['insol_huy06']['window_size'], cfgs['insol_huy06']['rollout_length_validation'])
X_trainb, Y_trainb, Insol_trainb = LPNE.create_windows(trainb, cfgs['insol_ber90']['window_size'], cfgs['insol_ber90']['rollout_length'])
X_valb, Y_valb, Insol_valb = LPNE.create_windows(valb, cfgs['insol_ber90']['window_size'], cfgs['insol_ber90']['rollout_length_validation'])
# decomposed: 3 forcing columns, so the state/forcing split is given explicitly (state_index = number of states)
X_trainc, Y_trainc, Insol_trainc = LPNE.create_windows(trainc, cfg_dec['window_size'], cfg_dec['rollout_length'], state_index=rc.n_state(cfg_dec))
X_valc, Y_valc, Insol_valc = LPNE.create_windows(valc, cfg_dec['window_size'], cfg_dec['rollout_length_validation'], state_index=rc.n_state(cfg_dec))

#print(X_traina)


### Training

##### using Huy06

In [ ]:
LPNE.set_seeds(cfgs['insol_huy06']['seed'])
braina = LPNE.CNN_model(cfgs['insol_huy06']['window_size'], params=paramsa, n_features=N_FEATURES)
braina.summary()

emua = AREmulator.RolloutEmulator(braina,
                                rollout_length=cfgs['insol_huy06']['rollout_length'],
                                vali_rollout_length=cfgs['insol_huy06']['rollout_length_validation'],
                                window_size=cfgs['insol_huy06']['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=paramsa['dwt gamma'],
                                dwt_beta=paramsa['dwt beta'],
                                dwt_eps=paramsa['dwt eps'],
                                dwt_levels=paramsa['dwt levels'],
                                )

emua.noise_level = cfgs['insol_huy06']['noise_level'] # this can be done when the class is instantiated
emua.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=paramsa['learning rate'], weight_decay=1e-3, clipnorm=1.0))

dummy_X = tf.zeros((1, cfgs['insol_huy06']['window_size'], N_FEATURES))
_ = emua(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfgs['insol_huy06'], scalea, splits_huy06)

checka = sc.PhysicalAttractorCheckpoint(
    filepath = pathsa['weights'],
    acf_threshold = cfgs['insol_huy06']['acf_threshold']
)
loggera = sc.WeightStatsLogger()

history = emua.fit(
    [X_traina, Insol_traina],
    Y_traina,
    validation_data=([X_vala, Insol_vala], Y_vala),
    epochs=cfgs['insol_huy06']['epochs'],
    callbacks=[checka, loggera],
    batch_size=cfgs['insol_huy06']['batch_size'],
    validation_batch_size=cfgs['insol_huy06']['batch_size'],
    shuffle=True,
)


##### using Ber90

In [ ]:
paramsb = cfgs['insol_ber90']['params']
pathsb = rc.run_paths(cfgs['insol_ber90']['run_tag'])
print("run tag :", cfgs['insol_ber90']['run_tag'])

LPNE.set_seeds(cfgs['insol_ber90']['seed'])
brainb = LPNE.CNN_model(cfgs['insol_ber90']['window_size'], params=paramsb, n_features=N_FEATURES)

emub = AREmulator.RolloutEmulator(brainb,
                                rollout_length=cfgs['insol_ber90']['rollout_length'],
                                vali_rollout_length=cfgs['insol_ber90']['rollout_length_validation'],
                                window_size=cfgs['insol_ber90']['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=paramsb['dwt gamma'],
                                dwt_beta=paramsb['dwt beta'],
                                dwt_eps=paramsb['dwt eps'],
                                dwt_levels=paramsb['dwt levels'],
                                noise_level=cfgs['insol_ber90']['noise_level'],
                                )
emub.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=paramsb['learning rate'], weight_decay=1e-1, clipnorm=1.0))

dummy_X = tf.zeros((1, cfgs['insol_ber90']['window_size'], N_FEATURES))
_ = emub(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfgs['insol_ber90'], scaleb, splits_ber90)

checkb = sc.PhysicalAttractorCheckpoint(
    filepath = pathsb['weights'],
    acf_threshold = cfgs['insol_ber90']['acf_threshold']
)
loggerb = sc.WeightStatsLogger()

historyb = emub.fit(
    [X_trainb, Insol_trainb],
    Y_trainb,
    validation_data=([X_valb, Insol_valb], Y_valb),
    epochs=cfgs['insol_ber90']['epochs'],
    callbacks=[checkb, loggerb],
    batch_size=cfgs['insol_ber90']['batch_size'],
    validation_batch_size=cfgs['insol_ber90']['batch_size'],
    shuffle=True,
)

##### using Ber90, decomposed (ecc, prec, obl)

In [ ]:
paramsc = cfg_dec['params']
pathsc = rc.run_paths(cfg_dec['run_tag'])
print("run tag :", cfg_dec['run_tag'])

LPNE.set_seeds(cfg_dec['seed'])
brainc = LPNE.CNN_model(cfg_dec['window_size'], params=paramsc, n_features=rc.n_features(cfg_dec))   # 6 inputs

emuc = AREmulator.RolloutEmulator(brainc,
                                rollout_length=cfg_dec['rollout_length'],
                                vali_rollout_length=cfg_dec['rollout_length_validation'],
                                window_size=cfg_dec['window_size'],
                                **rc.model_dims(cfg_dec),   # n_state=3, n_forcing=3 (ecc, prec, obl)
                                dwt_gamma=paramsc['dwt gamma'],
                                dwt_beta=paramsc['dwt beta'],
                                dwt_eps=paramsc['dwt eps'],
                                dwt_levels=paramsc['dwt levels'],
                                noise_level=cfg_dec['noise_level'],
                                )
emuc.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=paramsc['learning rate'], weight_decay=1e-1, clipnorm=1.0))

dummy_X = tf.zeros((1, cfg_dec['window_size'], rc.n_features(cfg_dec)))
_ = emuc(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfg_dec, scalec, splits_dec)

checkc = sc.PhysicalAttractorCheckpoint(
    filepath = pathsc['weights'],
    acf_threshold = cfg_dec['acf_threshold']
)
loggerc = sc.WeightStatsLogger()

historyc = emuc.fit(
    [X_trainc, Insol_trainc],
    Y_trainc,
    validation_data=([X_valc, Insol_valc], Y_valc),
    epochs=cfg_dec['epochs'],
    callbacks=[checkc, loggerc],
    batch_size=cfg_dec['batch_size'],
    validation_batch_size=cfg_dec['batch_size'],
    shuffle=True,
)

### Testing

In [ ]:
# boundary between the test period and the train/validation period (set by TRAIN_START; split_data skips
# the row in between), the same for both insolation records. Pass as plot_rollout(..., boundary=TEST_END)
# to mark it with a vertical line on every panel
TEST_END = (testt.min() + traint.max()) / 2

##### using Huy06

In [ ]:
print(f"Loading best physical model for final evaluation from {pathsa['weights']} ...")
emua.load_weights(str(pathsa['weights']))

best_epoch_a = checka.best_epoch
print(f"Best physical model found at epoch {best_epoch_a} with ACF threshold {cfgs['insol_huy06']['acf_threshold']}.")


In [ ]:
sp.LossPlots(history, checka)

In [ ]:
w = cfgs['insol_huy06']['window_size']

# rollout over the test period only (1461 ka to the training start)
rollout_a = sp.rollout(emua, testa, w)
sp.plot_rollout(emua, testa, testt, w, rollout_data = rollout_a)

# rollout seeded from the same oldest window, continued through train and validation to the present.
# Uses the full scaled record instead of joining the splits: `+` on numpy arrays adds them elementwise,
# and split_data leaves a 1-row gap between test and train, so joined splits would skip an age
fulla = scalea.transform(splits_huy06['full'])
fullt = splits_huy06['time']
rollout_full_a = sp.rollout(emua, fulla, w)
sp.plot_rollout(emua, fulla, fullt, w, rollout_data = rollout_full_a, boundary = TEST_END)

In [ ]:
# multitaper spectra of X (ice mass, dO18): the Ber90 extended rollout (seeded from the oldest test window and
# rolled out to the present) layered over the GPR mean of the full record it was rolled out against
spectra_b = sp.frequency_spectra(emua, fulla, w,
                                 timestep=cfgs['insol_ber90']['timestep'],
                                 spec_var=0,                    # 0 = X
                                 rollout_data=rollout_full_a)   # reuse the rollout from above instead of recomputing it
# multitaper spectra of X (ice mass, dO18): the Ber90 extended rollout (seeded from the oldest test window and
# rolled out to the present) layered over the GPR mean of the full record it was rolled out against
spectra_b = sp.frequency_spectra(emua, fulla, w,
                                 timestep=cfgs['insol_ber90']['timestep'],
                                 spec_var=1,                    # 0 = X
                                 rollout_data=rollout_full_a)   # reuse the rollout from above instead of recomputing it
# multitaper spectra of X (ice mass, dO18): the Ber90 extended rollout (seeded from the oldest test window and
# rolled out to the present) layered over the GPR mean of the full record it was rolled out against
spectra_b = sp.frequency_spectra(emua, fulla, w,
                                 timestep=cfgs['insol_ber90']['timestep'],
                                 spec_var=2,                    # 0 = X
                                 rollout_data=rollout_full_a)   # reuse the rollout from above instead of recomputing it


In [ ]:
# ---- Cross-correlation of ice mass (X, dO18) with the Ber90 insolation forcing ----
# Adapted from crossvalidation.ipynb (to move into a script later). Compares the ice-mass -> insolation lag in
# the GPR mean (ground truth) and in the Ber90 extended rollout, over the same ages, after the same filters.
#
# Two conventions to keep in mind:
#   * rows run OLDEST -> YOUNGEST (rc.load_data), so with scipy.signal.correlate(ice, forcing) a POSITIVE lag
#     means ice LAGS insolation (ice's pattern shows up later in time)
#   * ice and summer insolation are ANTI-correlated (strong 65N summer sun -> melting -> lower dO18), so the
#     insolation is sign-flipped before correlating. Otherwise the correlation peak lands on the in-phase
#     alignment, about half a cycle away from the physical response
from scipy import signal

def cross_correlate(x, y, dt, max_lag):
    '''
        normalized cross-correlation of x and y, restricted to |lag| <= max_lag (same units as dt), and the lag
        that maximizes it. scipy sign convention: a POSITIVE lag means x lags y
    '''
    x = x - x.mean()
    y = y - y.mean()
    corr = signal.correlate(x, y, mode="full") / (len(x) * x.std() * y.std())  # perfect match peaks near 1.0
    lags = signal.correlation_lags(len(x), len(y), mode="full") * dt

    mask = np.abs(lags) <= max_lag
    lags, corr = lags[mask], corr[mask]
    return lags, corr, lags[np.argmax(corr)]

def describe_lag(label, best_lag):
    if best_lag > 0:
        return f"{label}: ice LAGS insolation by {best_lag:.0f} kyr"
    if best_lag < 0:
        return f"{label}: ice LEADS insolation by {abs(best_lag):.0f} kyr"
    return f"{label}: in phase"

dt = cfgs['insol_ber90']['timestep']  # kyr
fs = 1.0 / dt

# drop the seed window: those rollout rows ARE the GPR values, so they'd inflate the agreement
no_seed = slice(w, None)
series = {   # label -> (ice, -insolation, color)
    "GPR mean (ground truth)": (fulla[no_seed, 0],          -fulla[no_seed, 3],          '#D6195E'),
    "Emulator rollout":        (rollout_full_a[no_seed, 0], -rollout_full_a[no_seed, 3], '#43A047'),
}

# filter -> (Butterworth sos, max |lag| searched [kyr]). sos + sosfiltfilt: zero-phase (no lag added by the
# filter itself) and numerically stable for these narrow bands. Band searches go to +/- half the period, since
# a band-limited correlation repeats every period
order = 4
FILTERS = {
    "lowpass (periods > 10 kyr)": (signal.butter(order, 0.1, btype="low", fs=fs, output="sos"), 100),
    "~100 kyr band (90-110 kyr)": (signal.butter(order, [1/110, 1/90], btype="band", fs=fs, output="sos"), 50),
    "~41 kyr band (40-42 kyr)":   (signal.butter(order, [1/42, 1/40], btype="band", fs=fs, output="sos"), 20.5),
    "~23 kyr band (21-25 kyr)":   (signal.butter(order, [1/25, 1/21], btype="band", fs=fs, output="sos"), 11.5),
}

xcorr_a = {}  # filter -> label -> (lags, corr, best_lag), kept for later use
with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(len(FILTERS), 1, figsize=(18, 5 * len(FILTERS)))

    for ax, (fname, (sos, max_lag)) in zip(axs, FILTERS.items()):
        xcorr_a[fname] = {}
        print(fname)
        for label, (ice, forcing, color) in series.items():
            lags, corr, best_lag = cross_correlate(signal.sosfiltfilt(sos, ice), signal.sosfiltfilt(sos, forcing),
                                                   dt, max_lag)
            xcorr_a[fname][label] = (lags, corr, best_lag)
            print("   " + describe_lag(label, best_lag) + f"  (r = {corr.max():.2f})")

            ax.plot(lags, corr, color=color, lw=2, label=label)
            ax.axvline(best_lag, color=color, linestyle='--', alpha=0.7, label=f"peak lag = {best_lag:.0f} kyr")

        ax.axvline(0, color='dimgray', lw=1)
        ax.set_title(f"X (ice mass) vs. $-$insolation (Huy06): {fname}", fontweight='bold')
        ax.set_ylabel("normalized cross-correlation")
        ax.set_xlabel("lag [kyr]  (> 0: ice lags insolation)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc='lower left', fontsize=13, ncol=2)

    plt.tight_layout()
    plt.show()


##### using Ber90

In [ ]:
print(f"Loading best physical model for final evaluation from {pathsb['weights']} ...")
emub.load_weights(str(pathsb['weights']))

best_epoch_b = checkb.best_epoch
print(f"Best physical model found at epoch {best_epoch_b} with ACF threshold {cfgs['insol_ber90']['acf_threshold']}.")

In [ ]:
sp.LossPlots(historyb, checkb)

In [ ]:
# X is ice mass, Y is CO2, and Z is deep ocean temperature
w = cfgs['insol_ber90']['window_size']

# rollout over the test period only (1461 ka to the training start)
rollout_b = sp.rollout(emub, testb, w)
sp.plot_rollout(emub, testb, testt, w, rollout_data = rollout_b)

# rollout seeded from the same oldest window, continued through train and validation to the present
fullb = scaleb.transform(splits_ber90['full'])
rollout_full_b = sp.rollout(emub, fullb, w, export_name = "rollout_full_b", time = fullt, scaler = scaleb)
sp.plot_rollout(emub, fullb, fullt, w, rollout_data = rollout_full_b, boundary = TEST_END)

In [ ]:
# multitaper spectra of X (ice mass, dO18): the Ber90 extended rollout (seeded from the oldest test window and
# rolled out to the present) layered over the GPR mean of the full record it was rolled out against
spectra_b = sp.frequency_spectra(emub, fullb, w,
                                 timestep=cfgs['insol_ber90']['timestep'],
                                 spec_var=0,                    # 0 = X
                                 rollout_data=rollout_full_b)   # reuse the rollout from above instead of recomputing it
# multitaper spectra of X (ice mass, dO18): the Ber90 extended rollout (seeded from the oldest test window and
# rolled out to the present) layered over the GPR mean of the full record it was rolled out against
spectra_b = sp.frequency_spectra(emub, fullb, w,
                                 timestep=cfgs['insol_ber90']['timestep'],
                                 spec_var=1,                    # 0 = X
                                 rollout_data=rollout_full_b)   # reuse the rollout from above instead of recomputing it
# multitaper spectra of X (ice mass, dO18): the Ber90 extended rollout (seeded from the oldest test window and
# rolled out to the present) layered over the GPR mean of the full record it was rolled out against
spectra_b = sp.frequency_spectra(emub, fullb, w,
                                 timestep=cfgs['insol_ber90']['timestep'],
                                 spec_var=2,                    # 0 = X
                                 rollout_data=rollout_full_b)   # reuse the rollout from above instead of recomputing it


In [ ]:
# ---- Cross-correlation of ice mass (X, dO18) with the Ber90 insolation forcing ----
# Adapted from crossvalidation.ipynb (to move into a script later). Compares the ice-mass -> insolation lag in
# the GPR mean (ground truth) and in the Ber90 extended rollout, over the same ages, after the same filters.
#
# Two conventions to keep in mind:
#   * rows run OLDEST -> YOUNGEST (rc.load_data), so with scipy.signal.correlate(ice, forcing) a POSITIVE lag
#     means ice LAGS insolation (ice's pattern shows up later in time)
#   * ice and summer insolation are ANTI-correlated (strong 65N summer sun -> melting -> lower dO18), so the
#     insolation is sign-flipped before correlating. Otherwise the correlation peak lands on the in-phase
#     alignment, about half a cycle away from the physical response
from scipy import signal

def cross_correlate(x, y, dt, max_lag):
    '''
        normalized cross-correlation of x and y, restricted to |lag| <= max_lag (same units as dt), and the lag
        that maximizes it. scipy sign convention: a POSITIVE lag means x lags y
    '''
    x = x - x.mean()
    y = y - y.mean()
    corr = signal.correlate(x, y, mode="full") / (len(x) * x.std() * y.std())  # perfect match peaks near 1.0
    lags = signal.correlation_lags(len(x), len(y), mode="full") * dt

    mask = np.abs(lags) <= max_lag
    lags, corr = lags[mask], corr[mask]
    return lags, corr, lags[np.argmax(corr)]

def describe_lag(label, best_lag):
    if best_lag > 0:
        return f"{label}: ice LAGS insolation by {best_lag:.0f} kyr"
    if best_lag < 0:
        return f"{label}: ice LEADS insolation by {abs(best_lag):.0f} kyr"
    return f"{label}: in phase"

dt = cfgs['insol_ber90']['timestep']  # kyr
fs = 1.0 / dt

# drop the seed window: those rollout rows ARE the GPR values, so they'd inflate the agreement
no_seed = slice(w, None)
series = {   # label -> (ice, -insolation, color)
    "GPR mean (ground truth)": (fullb[no_seed, 0],          -fullb[no_seed, 3],          '#D6195E'),
    "Emulator rollout":        (rollout_full_b[no_seed, 0], -rollout_full_b[no_seed, 3], '#43A047'),
}

# filter -> (Butterworth sos, max |lag| searched [kyr]). sos + sosfiltfilt: zero-phase (no lag added by the
# filter itself) and numerically stable for these narrow bands. Band searches go to +/- half the period, since
# a band-limited correlation repeats every period
order = 4
FILTERS = {
    "lowpass (periods > 10 kyr)": (signal.butter(order, 0.1, btype="low", fs=fs, output="sos"), 100),
    "~100 kyr band (90-110 kyr)": (signal.butter(order, [1/110, 1/90], btype="band", fs=fs, output="sos"), 50),
    "~41 kyr band (40-42 kyr)":   (signal.butter(order, [1/42, 1/40], btype="band", fs=fs, output="sos"), 20.5),
    "~23 kyr band (21-25 kyr)":   (signal.butter(order, [1/25, 1/21], btype="band", fs=fs, output="sos"), 11.5),
}

xcorr_b = {}  # filter -> label -> (lags, corr, best_lag), kept for later use
with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(len(FILTERS), 1, figsize=(18, 5 * len(FILTERS)))

    for ax, (fname, (sos, max_lag)) in zip(axs, FILTERS.items()):
        xcorr_b[fname] = {}
        print(fname)
        for label, (ice, forcing, color) in series.items():
            lags, corr, best_lag = cross_correlate(signal.sosfiltfilt(sos, ice), signal.sosfiltfilt(sos, forcing),
                                                   dt, max_lag)
            xcorr_b[fname][label] = (lags, corr, best_lag)
            print("   " + describe_lag(label, best_lag) + f"  (r = {corr.max():.2f})")

            ax.plot(lags, corr, color=color, lw=2, label=label)
            ax.axvline(best_lag, color=color, linestyle='--', alpha=0.7, label=f"peak lag = {best_lag:.0f} kyr")

        ax.axvline(0, color='dimgray', lw=1)
        ax.set_title(f"X (ice mass) vs. $-$insolation (Ber90): {fname}", fontweight='bold')
        ax.set_ylabel("normalized cross-correlation")
        ax.set_xlabel("lag [kyr]  (> 0: ice lags insolation)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc='lower left', fontsize=13, ncol=2)

    plt.tight_layout()
    plt.show()


##### using Ber90, decomposed (ecc, prec, obl)

In [ ]:
print(f"Loading best physical model for final evaluation from {pathsc['weights']} ...")
emuc.load_weights(str(pathsc['weights']))

best_epoch_c = checkc.best_epoch
print(f"Best physical model found at epoch {best_epoch_c} with ACF threshold {cfg_dec['acf_threshold']}.")

In [ ]:
sp.LossPlots(historyc, checkc)

In [ ]:
# X is ice mass, Y is CO2, and Z is deep ocean temperature; forced by ecc, prec, obl instead of one insolation column
w = cfg_dec['window_size']

# rollout over the test period only (1461 ka to the training start)
rollout_c = sp.rollout(emuc, testc, w)
sp.plot_rollout(emuc, testc, testt, w, rollout_data = rollout_c, forcing_names = rc.forcing_names(cfg_dec))

# rollout seeded from the same oldest window, continued through train and validation to the present
fullc = scalec.transform(splits_dec['full'])
rollout_full_c = sp.rollout(emuc, fullc, w)
sp.plot_rollout(emuc, fullc, fullt, w, rollout_data = rollout_full_c, boundary = TEST_END,
                forcing_names = rc.forcing_names(cfg_dec))

In [ ]:
# multitaper spectra of X, Y and Z: the decomposed-Ber90 extended rollout (seeded from the oldest test window and
# rolled out to the present) layered over the GPR mean of the full record it was rolled out against
spectra_c = {}
for k, name in enumerate(rc.state_names(cfg_dec)):   # 0 = X, 1 = Y, 2 = Z
    spectra_c[name] = sp.frequency_spectra(emuc, fullc, w,
                                           timestep=cfg_dec['timestep'],
                                           spec_var=k,
                                           rollout_data=rollout_full_c)   # reuse the rollout from above

In [ ]:
# ---- Cross-correlation of ice mass (X, dO18) with the Ber90 insolation, for the decomposed-Ber90 model ----
# Same analysis as for Ber90 above (uses cross_correlate, describe_lag, FILTERS, dt from that cell, so run it
# first). This model is forced by ecc, prec and obl, not by one insolation column, so its ice rollout is
# correlated with the same Ber90 65N solstice insolation as the Ber90 model's (taken from that record, fullb[:, 3],
# over the same ages): the lags of both models are then measured against the same reference.
# Insolation is sign-flipped as above (ice and summer insolation are anti-correlated)
no_seed = slice(w, None)
insol_ref = -fullb[no_seed, 3]
series = {   # label -> (ice, -insolation, color)
    "GPR mean (ground truth)": (fullc[no_seed, 0],          insol_ref, '#D6195E'),
    "Emulator rollout":        (rollout_full_c[no_seed, 0], insol_ref, '#43A047'),
}

xcorr_c = {}  # filter -> label -> (lags, corr, best_lag), kept for later use
with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(len(FILTERS), 1, figsize=(18, 5 * len(FILTERS)))

    for ax, (fname, (sos, max_lag)) in zip(axs, FILTERS.items()):
        xcorr_c[fname] = {}
        print(fname)
        for label, (ice, forcing, color) in series.items():
            lags, corr, best_lag = cross_correlate(signal.sosfiltfilt(sos, ice), signal.sosfiltfilt(sos, forcing),
                                                   dt, max_lag)
            xcorr_c[fname][label] = (lags, corr, best_lag)
            print("   " + describe_lag(label, best_lag) + f"  (r = {corr.max():.2f})")

            ax.plot(lags, corr, color=color, lw=2, label=label)
            ax.axvline(best_lag, color=color, linestyle='--', alpha=0.7, label=f"peak lag = {best_lag:.0f} kyr")

        ax.axvline(0, color='dimgray', lw=1)
        ax.set_title(f"X (ice mass) vs. $-$insolation (Ber90), decomposed-Ber90 model: {fname}", fontweight='bold')
        ax.set_ylabel("normalized cross-correlation")
        ax.set_xlabel("lag [kyr]  (> 0: ice lags insolation)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc='lower left', fontsize=13, ncol=2)

    plt.tight_layout()
    plt.show()

### Comparison

In [ ]:
# X (ice mass, benthic d18O) rollout of each model over the ground truth, one panel per insolation forcing.
# Full-record rollouts (seeded from the oldest window, rolled out to the present) from the testing sections above.
# Plotted in physical units [permil] so the panels share one y-axis; the L2 in each title is in that model's scaled
# units, as in plot_rollout (seed window excluded; "test" = rows before the test | train boundary)
COMPARE = [   # (title, rollout, scaled ground truth, scaler)
    ("Huybers 2006 ISI",                        rollout_full_a, fulla, scalea),
    ("Berger 1990 65N June solstice",           rollout_full_b, fullb, scaleb),
    ("Berger 1990 decomposed (ecc, prec, obl)", rollout_full_c, fullc, scalec),
]
COLOR_TRUTH, COLOR_EMU = "#D6195E", "#FFB600"   # X's (data, emulator) colors in sp.plot_rollout
n_test = len(testt)                             # leading rows of the record that are the test set

with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(len(COMPARE), 1, figsize=(22, 4.5 * len(COMPARE)), sharex=True, sharey=True)

    for ax, (title, ro, truth, scaler) in zip(axs, COMPARE):
        to_phys = lambda x: x * scaler.scale_[0] + scaler.mean_[0]   # X column back to d18O [permil]
        n = ro.shape[0]
        ax.plot(fullt[:n], to_phys(truth[:n, 0]), color=COLOR_TRUTH, linestyle='dashed', lw=2, alpha=0.6,
                label="GPR mean (ground truth)")
        ax.plot(fullt[:n], to_phys(ro[:, 0]), color=COLOR_EMU, lw=2, label="emulator rollout")

        l2_all = mean_squared_error(truth[w:n, 0], ro[w:, 0])
        l2_test = mean_squared_error(truth[w:n_test, 0], ro[w:n_test, 0])
        ax.set_title(f"{title}:   " + r"$\mathcal{L}_{L_2}$" + f" = {l2_all:.3f}   (test set: {l2_test:.3f})")
        ax.axvline(TEST_END, color='black', linestyle=':', lw=2)
        ax.set_ylabel(r"X: $\delta^{18}$O [‰]")
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper left', fontsize=14)

    axs[0].annotate('test  |  train / validation', xy=(TEST_END, 1.0), xycoords=('data', 'axes fraction'),
                    xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14)
    axs[-1].invert_xaxis()   # shared x-axis: oldest on the left
    axs[-1].set_xlabel("time [kyr]")
    plt.tight_layout()
    plt.show()